# Q1. Ingestion Snowflake

Chargement des fichiers WAV dans un Stage Snowflake et stocker les métadonnées dans une table structurée.


In [ ]:
import os
import librosa
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from snowflake.snowpark import Session
from snowflake.snowpark.types import (
    StructType, StructField,
    StringType, FloatType, IntegerType, LongType, TimestampType
)

load_dotenv('../.env', override=True)
print('Librairies chargées.')

In [ ]:
DATA_DIR      = '../data'
MIN_DURATION  = 1.0       # secondes — drop fichiers trop courts
STAGE_NAME    = 'RESPIRATORY_STAGE'
TABLE_META    = 'AUDIO_METADATA'
TABLE_PRED    = 'PREDICTIONS'

## Connexion Snowflake

In [ ]:
import getpass

totp = getpass.getpass("Code MFA (6 chiffres) : ")

connection_params = {
    'account'      : os.environ['SNOWFLAKE_ACCOUNT'],
    'user'         : os.environ['SNOWFLAKE_USER'],
    'password'     : os.environ['SNOWFLAKE_PASSWORD'],
    'authenticator': os.environ['SNOWFLAKE_AUTHENTICATOR'],
    'passcode'     : totp,
    'warehouse'    : os.environ['SNOWFLAKE_WAREHOUSE'],
    'database'     : os.environ['SNOWFLAKE_DATABASE'],
    'schema'       : os.environ['SNOWFLAKE_SCHEMA'],
    'role'         : os.environ['SNOWFLAKE_ROLE'],
}

session = Session.builder.configs(connection_params).create()
print(f"Connecté : {session.get_current_database()}.{session.get_current_schema()}")
print(f"Warehouse : {session.get_current_warehouse()}")

## Création du Stage et des Tables

In [ ]:
session.sql(f"CREATE STAGE IF NOT EXISTS {STAGE_NAME} COMMENT = 'Fichiers WAV bruts — Asthma Detection Dataset v2'").collect()

session.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_META} (
        FILENAME      VARCHAR(255) NOT NULL,
        LABEL         VARCHAR(50)  NOT NULL,
        DURATION_S    FLOAT        NOT NULL,
        SAMPLE_RATE   INTEGER      NOT NULL,
        N_SAMPLES     INTEGER      NOT NULL,
        FILE_SIZE_KB  FLOAT,
        STAGE_PATH    VARCHAR(500)
    )
""").collect()

session.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_PRED} (
        PREDICTION_ID  VARCHAR(36)   DEFAULT UUID_STRING(),
        TIMESTAMP      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
        PHARMACIE_ID   VARCHAR(100)  NOT NULL,
        CLASSE_PREDITE VARCHAR(50)   NOT NULL,
        PROBABILITES   VARIANT,
        CONFIANCE      FLOAT         NOT NULL
    )
""").collect()

print(f"Stage  '{STAGE_NAME}' : OK")
print(f"Table  '{TABLE_META}' : OK")
print(f"Table  '{TABLE_PRED}' : OK")

## Extraction des métadonnées locales

In [ ]:
records = []
skipped = 0

for label in sorted(os.listdir(DATA_DIR)):
    label_dir = os.path.join(DATA_DIR, label)
    if not os.path.isdir(label_dir):
        continue
    for fname in sorted(os.listdir(label_dir)):
        if not fname.lower().endswith('.wav'):
            continue
        fpath = os.path.join(label_dir, fname)
        duration = librosa.get_duration(path=fpath)
        if duration < MIN_DURATION:
            print(f"  Skipped (trop court) : {fname}")
            skipped += 1
            continue
        sr_native = librosa.get_samplerate(fpath)
        n_samples  = int(duration * sr_native)
        size_kb    = os.path.getsize(fpath) / 1024
        stage_path = f'@{STAGE_NAME}/{label.lower()}/{fname}'
        records.append({
            'FILENAME'    : fname,
            'LABEL'       : label.lower(),
            'DURATION_S'  : round(duration, 4),
            'SAMPLE_RATE' : sr_native,
            'N_SAMPLES'   : n_samples,
            'FILE_SIZE_KB': round(size_kb, 2),
            'STAGE_PATH'  : stage_path,
        })

df_meta = pd.DataFrame(records)
print(f"Fichiers indexés : {len(df_meta)} | Skippés : {skipped}")
df_meta.groupby('LABEL')[['DURATION_S', 'SAMPLE_RATE']].describe().round(2)

## Upload WAV dans le Stage Snowflake

In [ ]:
from tqdm.auto import tqdm

errors = []

for label in sorted(os.listdir(DATA_DIR)):
    label_dir = os.path.join(DATA_DIR, label)
    if not os.path.isdir(label_dir):
        continue

    wav_files = [f for f in os.listdir(label_dir) if f.lower().endswith('.wav')]
    print(f"\n[{label.lower()}] {len(wav_files)} fichiers...")

    for fname in tqdm(wav_files, desc=label):
        fpath    = Path(label_dir) / fname
        duration = librosa.get_duration(path=str(fpath))
        if duration < MIN_DURATION:
            continue
        try:
            put_result = session.file.put(
                local_file_name = str(fpath.resolve()),
                stage_location  = f'@{STAGE_NAME}/{label.lower()}/',
                auto_compress   = False,
                overwrite       = False,
            )
        except Exception as e:
            errors.append((fname, str(e)))

print(f"\nUpload terminé | Erreurs : {len(errors)}")
if errors:
    for f, e in errors:
        print(f"  {f} → {e}")

## Insertion des métadonnées dans Snowflake

In [ ]:
schema = StructType([
    StructField('FILENAME',     StringType()),
    StructField('LABEL',        StringType()),
    StructField('DURATION_S',   FloatType()),
    StructField('SAMPLE_RATE',  IntegerType()),
    StructField('N_SAMPLES',    IntegerType()),
    StructField('FILE_SIZE_KB', FloatType()),
    StructField('STAGE_PATH',   StringType()),
])

sp_df = session.create_dataframe(df_meta[[
    'FILENAME', 'LABEL', 'DURATION_S',
    'SAMPLE_RATE', 'N_SAMPLES', 'FILE_SIZE_KB', 'STAGE_PATH'
]].values.tolist(), schema=schema)

sp_df.write.mode('overwrite').save_as_table(TABLE_META)
print(f"Inséré {len(df_meta)} lignes dans {TABLE_META}")

## Vérification

In [ ]:
# Résumé par classe
result = session.sql(f"""
    SELECT
        LABEL,
        COUNT(*)              AS N_FILES,
        ROUND(AVG(DURATION_S), 2) AS AVG_DURATION_S,
        MIN(SAMPLE_RATE)      AS SR_MIN,
        MAX(SAMPLE_RATE)      AS SR_MAX,
        ROUND(SUM(FILE_SIZE_KB)/1024, 1) AS TOTAL_MB
    FROM {TABLE_META}
    GROUP BY LABEL
    ORDER BY N_FILES DESC
""").to_pandas()

print(f"\n=== {TABLE_META} ===")
print(result.to_string(index=False))
print(f"\nTotal : {result['N_FILES'].sum()} fichiers, {result['TOTAL_MB'].sum():.1f} MB")

In [ ]:
# Vérif Stage
stage_files = session.sql(f"LIST @{STAGE_NAME}").to_pandas()
print(f"Fichiers dans le Stage : {len(stage_files)}")
stage_files.head(10)

In [ ]:
session.close()
print("Session Snowflake fermée.")

## Upload du modèle entraîné → Stage Snowflake

In [ ]:
import getpass
from pathlib import Path
from tqdm.auto import tqdm

totp = getpass.getpass("Code MFA (6 chiffres) : ")

connection_params = {
    'account'      : os.environ['SNOWFLAKE_ACCOUNT'],
    'user'         : os.environ['SNOWFLAKE_USER'],
    'password'     : os.environ['SNOWFLAKE_PASSWORD'],
    'authenticator': os.environ['SNOWFLAKE_AUTHENTICATOR'],
    'passcode'     : totp,
    'warehouse'    : os.environ['SNOWFLAKE_WAREHOUSE'],
    'database'     : os.environ['SNOWFLAKE_DATABASE'],
    'schema'       : os.environ['SNOWFLAKE_SCHEMA'],
    'role'         : os.environ['SNOWFLAKE_ROLE'],
}

session = Session.builder.configs(connection_params).create()
print(f"Connecté : {session.get_current_database()}.{session.get_current_schema()}")

model_path = Path('../models/best_model_convnext.pth').resolve()
size_mb = model_path.stat().st_size / 1024 / 1024
print(f"Upload {model_path.name} ({size_mb:.1f} MB)...")

session.file.put(
    local_file_name = str(model_path),
    stage_location  = f'@{STAGE_NAME}/models/',
    auto_compress   = False,
    overwrite       = True,
)
print(f"✓ {model_path.name} uploadé dans @{STAGE_NAME}/models/")

session.close()
print("Session fermée.")